In [9]:

import pandas as pd
import numpy as np
 
# ── CONFIG ────────────────────────────────────────────────────────────────────
START = '2014-01-01'
END   = '2024-12-31'
 
COUNTRIES = [
    'Brazil', 'Chile', 'China', 'Colombia', 'Indonesia',
    'Malaysia', 'Mexico', 'Philippines', 'Qatar', 'South Africa',
    'South Korea', 'Saudi Arabia', 'Turkey', 'Thailand',
    'United Arab Emirates', 'Egypt'
]
UAE_SPLIT = ['Abu Dhabi', 'Dubai']

DATA = '../data/processed/CCA_V2/'

In [10]:
# ── STEP 1: Master date index from CDS ───────────────────────────────────────
cds_raw = pd.read_csv(DATA + 'cds.csv', index_col=0, parse_dates=True)
cds_raw.index = pd.to_datetime(cds_raw.index)
cds_raw = cds_raw.loc[START:END]
master = cds_raw.index.sort_values()
 
# CDS: keep our countries
cds_countries = COUNTRIES + UAE_SPLIT
cds = cds_raw[[c for c in cds_countries if c in cds_raw.columns]]

In [11]:
# ── STEP 2: Exchange rates (daily, DD.MM.YYYY) ───────────────────────────────
fx_raw = pd.read_csv(DATA + 'exchange_rates.csv', index_col=0)
fx_raw.index = pd.to_datetime(fx_raw.index, format='%d.%m.%Y', dayfirst=True)
fx_raw = fx_raw.sort_index()
fx = fx_raw[[c for c in COUNTRIES if c in fx_raw.columns]]
fx = fx.reindex(master, method='ffill')

In [12]:
# ── STEP 3: Interest rates (monthly, DD.MM.YYYY) ─────────────────────────────
rates_raw = pd.read_csv(DATA + 'interest_rates.csv', index_col=0)
rates_raw.index = pd.to_datetime(rates_raw.index, format='%d.%m.%Y', dayfirst=True)
rates_raw = rates_raw.sort_index()
 
# US risk-free rate
us_rate = rates_raw[['United States']].reindex(
    pd.date_range(rates_raw.index.min(), master.max(), freq='D')
).ffill().reindex(master, method='ffill')

# Domestic rates
dom_rate = rates_raw[[c for c in COUNTRIES if c in rates_raw.columns]]
dom_rate = dom_rate.reindex(
    pd.date_range(dom_rate.index.min(), master.max(), freq='D')
).ffill().reindex(master, method='ffill')

In [13]:
# ── STEP 4: Monetary base (monthly, DD.MM.YYYY, millions local currency) ─────
mb_raw = pd.read_csv(DATA + 'monetary_agg.csv', index_col=0)
BILLIONS_MULT = 1000  # billions → millions
THOUSANDS_MULT = 1 / 1000  # thousands → millions

mb_raw.index = pd.to_datetime(mb_raw.index, format='%d.%m.%Y', dayfirst=True)
mb_raw = mb_raw.sort_index()

# China: hundreds of millions → millions

for col in mb_raw.columns:
    if col in ['China', 'Indonesia', 'Colombia', 'Chile']:
        mb_raw[col] = mb_raw[col] * BILLIONS_MULT
    elif col=='Turkey':
        mb_raw[col] = mb_raw[col] * THOUSANDS_MULT
    else:
        mb_raw[col] = mb_raw[col]

mb = mb_raw[[c for c in COUNTRIES if c in mb_raw.columns]]
mb = mb.reindex(
    pd.date_range(mb_raw.index.min(), master.max(), freq='D')
).ffill().reindex(master, method='ffill')

In [14]:
# ── STEP 5: External debt (annual, integer year index, millions USD) ──────────


ext_raw = pd.read_csv(DATA + 'external_debt.csv', index_col=0)
# integer year → Dec 31 of that year
ext_raw.index = pd.to_datetime(ext_raw.index.astype(str) + '-12-31')
ext_raw = ext_raw.sort_index()

ext = ext_raw[[c for c in COUNTRIES if c in ext_raw.columns]]
ext = ext.reindex(
    pd.date_range(ext_raw.index.min(), master.max(), freq='D')
).ffill().reindex(master, method='ffill')

In [15]:
# ── STEP 6: Domestic debt (net of central bank claims on gov)
# domestic_debt.csv (quarterly) minus cnbk_claims_on_gov.csv (monthly)
# All values normalised to millions local currency before subtracting

BILLIONS_COUNTRIES = {'China', 'Chile', 'Indonesia', 'Mexico', 'South Korea', 'Thailand', 'Colombia'}

# ── 6a: Load domestic debt (quarterly) ───────────────────────────────────────
def parse_quarter(q):
    quarter, year = q.split()
    month_end = {'Q1': 3, 'Q2': 6, 'Q3': 9, 'Q4': 12}[quarter]
    return pd.Timestamp(year=int(year), month=month_end, day=1) + pd.offsets.MonthEnd(0)

dd_raw = pd.read_csv(DATA + 'domestic_debt.csv')
dd_raw.index = dd_raw['Quarter'].map(parse_quarter)
dd_raw = dd_raw.drop(columns='Quarter').sort_index()

# Billions → millions
for col in dd_raw.columns:
    if col in BILLIONS_COUNTRIES:
        dd_raw[col] = dd_raw[col] * 1000

# ── 6b: Load cnbk claims on gov (monthly) ────────────────────────────────────
cnbk_raw = pd.read_csv(DATA + 'cnbk_claims_on_gov.csv', index_col=0)
cnbk_raw.index = pd.to_datetime(cnbk_raw.index, format='%d.%m.%Y', dayfirst=True)
cnbk_raw = cnbk_raw.sort_index()

# China: hundreds of millions → millions
if 'China' in cnbk_raw.columns:
    cnbk_raw['China'] = cnbk_raw['China'] * 100

# ── 6c: Align to daily, subtract, reindex to master ──────────────────────────
full_range = pd.date_range(
    min(dd_raw.index.min(), cnbk_raw.index.min()),
    master.max(),
    freq='D'
)

dd_daily   = dd_raw.reindex(full_range).ffill()
cnbk_daily = cnbk_raw.reindex(full_range).ffill()

# Subtract on shared countries only; fall back to dd alone if cnbk missing
dom_raw = dd_daily.copy()
shared = dd_daily.columns.intersection(cnbk_daily.columns)
dom_raw[shared] = dd_daily[shared].subtract(cnbk_daily[shared])

dom = dom_raw[[c for c in COUNTRIES if c in dom_raw.columns]]
dom = dom.reindex(master, method='ffill')

In [16]:
# ── STEP 7: Build long panel ──────────────────────────────────────────────────
rows = []
for country in COUNTRIES:
    cds_col = country  # will handle UAE below
    for date in master:
        row = {'date': date, 'country': country}
 
        # CDS
        if country == 'United Arab Emirates':
            ad = cds.get('Abu Dhabi', pd.Series(dtype=float))
            db = cds.get('Dubai', pd.Series(dtype=float))
            row['cds_spread'] = ad.get(date, np.nan) if date in ad.index else np.nan
            row['country'] = 'UAE (Abu Dhabi)'
        else:
            row['cds_spread'] = cds[country].get(date, np.nan) if country in cds.columns else np.nan
 
        # FX
        row['fx_rate'] = fx[country].get(date, np.nan) if country in fx.columns else np.nan
 
        # Rates (convert % to decimal)
        row['domestic_rate'] = (dom_rate[country].get(date, np.nan) / 100
                                if country in dom_rate.columns else np.nan)
        row['risk_free_rate'] = us_rate['United States'].get(date, np.nan) / 100
 
        # Monetary base (millions local currency)
        row['monetary_base_bn_local'] = mb[country].get(date, np.nan)/1000 if country in mb.columns else np.nan
 
        # External debt (millions USD → billions USD)
        row['external_debt_bn_usd'] = (ext[country].get(date, np.nan) / 1000
                                       if country in ext.columns else np.nan)
 
        # Domestic debt (billions local currency)
        row['domestic_debt_bn_local'] = dom[country].get(date, np.nan) / 1000 if country in dom.columns else np.nan 
        rows.append(row)
 
panel = pd.DataFrame(rows)

panel.to_csv('../data/processed/CCA_V2/CCA_panel.csv', index=False)

In [19]:
# ── STEP 9: Sanity check ──────────────────────────────────────────────────────
panel['dom_debt_usd'] = panel['domestic_debt_bn_local'] / panel['fx_rate']
panel['mon_base_usd'] = panel['monetary_base_bn_local'] / panel['fx_rate']
panel['fx_vol'] = panel.groupby('country')['fx_rate'].transform(lambda x: np.log(x).diff().std() * np.sqrt(52))

check = panel.groupby('country').agg(
    fx_vol=('fx_vol','mean'),
    cds_spread=('cds_spread','mean'),
    dom_debt_usd_bn=('dom_debt_usd','mean'),
    ext_debt_usd_bn=('external_debt_bn_usd','mean'),
    mon_base_usd_bn=('mon_base_usd','mean'),
).round(2)


check['DtE Ratio'] = (check['ext_debt_usd_bn'] / (check['dom_debt_usd_bn'] + check['mon_base_usd_bn'] )).round(2)

check

,fx_vol,cds_spread,dom_debt_usd_bn,ext_debt_usd_bn,mon_base_usd_bn,DtE Ratio
country,,,,,,
Brazil,0.15,214.37,818.56,556.03,77.88,0.62
Chile,0.13,75.18,47.96,190.76,58.00,1.80
China,0.04,70.05,5398.01,1983.95,1209.15,0.30
Colombia,0.14,163.44,160.33,156.95,20.87,0.87
Egypt,0.25,537.35,181.45,99.99,48.34,0.44
Indonesia,0.07,121.09,163.90,359.29,44.49,1.72
Malaysia,0.07,84.63,142.47,228.95,25.76,1.36
Mexico,0.13,121.28,382.20,575.88,97.28,1.20
Philippines,0.05,79.45,101.59,89.61,24.96,0.71
